In [ ]:
# CELL 1 SEtup Paths

from pathlib import Path  # Handles Windows paths safely.
import zipfile  # Reads .zip files without extracting the full archive.
import shutil  # Copies only needed nested zip files to a temporary folder.
import io  # Lets pandas read CSV bytes from inside zip files.
import re  # Cleans Excel sheet names.
import hashlib  # Creates short unique names for temporary nested zip files.
import pandas as pd  # Reads CSV files and writes Excel workbooks.
from IPython.display import display  # Displays tables inside Jupyter.

project_root = Path(r"Z:\Research-analysis")  # Main project folder on the remote Windows computer.

notebook_name = "Data-Excel"  # Name used for the output folder.

run_name = "Run_G0016"  # Change this later to Run_G0002, Run_G0003, etc.

transfer_zip = project_root / "data" / "raw" / "LUTO2-Scenario-Run-2" / "transfer_3573951_files_89a7061e.zip"  # Main LUTO2 zip file.

output_folder = project_root / "data" / "processed" / notebook_name  # Folder where the consolidated Excel file will be saved.

temporary_folder = output_folder / "_temporary_run_archives"  # Temporary folder for nested zip files only.

output_folder.mkdir(parents=True, exist_ok=True)  # Creates the output folder if needed.

temporary_folder.mkdir(parents=True, exist_ok=True)  # Creates the temporary folder if needed.

output_excel = output_folder / f"{run_name}_csv_consolidated.xlsx"  # Final Excel workbook path.

include_report_data_zip = False  # Keep False for only Run_G0001 folder; set True if you also want Report_Data/Run_G0001.zip.

keep_temporary_archives = False  # Deletes temporary nested zip copies after processing.

print("Output workbook will be saved to:", output_excel)  # Confirms output location.

In [2]:
# CELL 2 HElper functions

max_excel_rows = 1_048_576  # Excel's maximum number of rows per sheet.

max_excel_columns = 16_384  # Excel's maximum number of columns per sheet.

def make_safe_sheet_name(file_name, used_sheet_names):  # Creates a valid unique Excel sheet name.
    base_name = Path(file_name).stem  # Uses the CSV filename without .csv.
    clean_name = re.sub(r"[\[\]\:\*\?\/\\]", "_", base_name)  # Replaces characters Excel does not allow.
    clean_name = re.sub(r"\s+", " ", clean_name).strip()  # Removes repeated spaces.
    clean_name = clean_name or "Sheet"  # Uses a fallback name if the filename becomes empty.
    clean_name = clean_name[:31]  # Excel sheet names cannot exceed 31 characters.
    candidate = clean_name  # Starts with the cleaned filename.
    counter = 1  # Counter used if duplicate sheet names occur.

    while candidate.lower() in used_sheet_names:  # Checks duplicate sheet names case-insensitively.
        suffix = f"_{counter:02d}"  # Creates suffix like _01, _02.
        candidate = clean_name[:31 - len(suffix)] + suffix  # Keeps total sheet name within 31 characters.
        counter += 1  # Moves to the next suffix if needed.

    used_sheet_names.add(candidate.lower())  # Records the sheet name as used.
    return candidate  # Returns the safe Excel sheet name.

def read_csv_member(zip_object, member_name):  # Reads one CSV file from a zip archive.
    csv_bytes = zip_object.read(member_name)  # Reads the CSV file as bytes from inside the zip.

    for encoding in ["utf-8-sig", "utf-8", "cp1252", "latin1"]:  # Tries common CSV encodings.
        try:
            table = pd.read_csv(io.BytesIO(csv_bytes), encoding=encoding, low_memory=False)  # Converts CSV bytes to a DataFrame.
            return table, encoding  # Returns the DataFrame and successful encoding.
        except UnicodeDecodeError:
            continue  # Tries the next encoding if text decoding fails.

    table = pd.read_csv(io.BytesIO(csv_bytes), encoding="latin1", low_memory=False)  # Final fallback encoding.
    return table, "latin1"  # Returns the fallback result.

def copy_zip_member_to_temp(zip_object, member_name, container_label):  # Copies a nested zip file to disk temporarily.
    digest = hashlib.md5(f"{container_label}!{member_name}".encode()).hexdigest()[:10]  # Creates a short unique ID.
    safe_stem = re.sub(r"[^A-Za-z0-9._-]+", "_", Path(member_name).stem)[:80]  # Makes a safe temporary filename.
    temp_zip_path = temporary_folder / f"{safe_stem}_{digest}.zip"  # Creates the temporary zip path.

    with zip_object.open(member_name) as source_file:  # Opens the nested zip file inside the parent zip.
        with open(temp_zip_path, "wb") as target_file:  # Opens a temporary file on disk.
            shutil.copyfileobj(source_file, target_file, length=16 * 1024 * 1024)  # Copies in 16 MB chunks.

    return temp_zip_path  # Returns the temporary zip path.

def write_csv_to_workbook(zip_object, member_name, source_location, writer, audit_rows, used_sheet_names):  # Writes one CSV as one Excel sheet.
    sheet_name = make_safe_sheet_name(Path(member_name).name, used_sheet_names)  # Creates the sheet name from the CSV filename.

    try:
        table, encoding = read_csv_member(zip_object, member_name)  # Reads the CSV into pandas.
        row_count = len(table)  # Counts rows.
        column_count = len(table.columns)  # Counts columns.

        if row_count > max_excel_rows or column_count > max_excel_columns:  # Checks Excel size limits.
            status = "SKIPPED_TOO_LARGE"  # Marks file as skipped if Excel cannot store it.
            message = "CSV exceeds Excel worksheet size limit."  # Explains why skipped.
        else:
            table.to_excel(writer, sheet_name=sheet_name, index=False)  # Saves the CSV table as one Excel sheet.
            status = "WRITTEN"  # Marks file as successfully written.
            message = ""  # No error message.

        audit_rows.append({  # Records file-level audit information.
            "status": status,
            "csv_file": Path(member_name).name,
            "excel_sheet": sheet_name,
            "source_location": source_location,
            "rows": row_count,
            "columns": column_count,
            "encoding": encoding,
            "message": message,
        })

        print(f"{status}: {Path(member_name).name} -> {sheet_name} ({row_count} rows)")  # Progress message.

    except Exception as error:
        audit_rows.append({  # Records any read/write error.
            "status": "ERROR",
            "csv_file": Path(member_name).name,
            "excel_sheet": sheet_name,
            "source_location": source_location,
            "rows": None,
            "columns": None,
            "encoding": None,
            "message": str(error),
        })

        print(f"ERROR: {Path(member_name).name} -> {error}")  # Prints the error.

def process_zip_file(zip_path, container_label, writer, audit_rows, used_sheet_names, depth=0, max_depth=3):  # Processes CSVs inside a zip file.
    with zipfile.ZipFile(zip_path, "r") as current_zip:  # Opens the zip file from disk.
        members = current_zip.infolist()  # Lists files inside the zip.

        csv_members = [m.filename for m in members if not m.is_dir() and m.filename.lower().endswith(".csv")]  # Finds CSV files.

        nested_zip_members = [m.filename for m in members if not m.is_dir() and m.filename.lower().endswith(".zip")]  # Finds nested zip files.

        for csv_member in csv_members:  # Loops through CSV files.
            source_location = f"{container_label}!{csv_member}"  # Records exact source path inside zip.
            write_csv_to_workbook(current_zip, csv_member, source_location, writer, audit_rows, used_sheet_names)  # Writes CSV to Excel.

        if depth < max_depth:  # Prevents infinite recursion if archives are unexpectedly nested.
            for nested_zip_member in nested_zip_members:  # Loops through nested zip files.
                nested_label = f"{container_label}!{nested_zip_member}"  # Records nested zip source.
                nested_temp_path = copy_zip_member_to_temp(current_zip, nested_zip_member, nested_label)  # Copies nested zip to temp disk file.

                try:
                    process_zip_file(nested_temp_path, nested_label, writer, audit_rows, used_sheet_names, depth + 1, max_depth)  # Processes the nested zip.
                finally:
                    if not keep_temporary_archives and nested_temp_path.exists():  # Checks whether temporary zip should be deleted.
                        nested_temp_path.unlink()  # Deletes the temporary nested zip.

In [ ]:
# CELL 3 Locate run folder csv files

with zipfile.ZipFile(transfer_zip, "r") as outer_zip:  # Opens the main LUTO2 transfer zip.
    outer_members = [m.filename for m in outer_zip.infolist() if not m.is_dir()]  # Lists files, excluding folders.

run_prefix = f"{run_name}/"  # Prefix used for files inside the selected run folder.

direct_csv_members = [m for m in outer_members if m.startswith(run_prefix) and m.lower().endswith(".csv")]  # Finds direct CSVs in Run_G0001.

nested_zip_members = [m for m in outer_members if m.startswith(run_prefix) and m.lower().endswith(".zip")]  # Finds zip files inside Run_G0001.

report_zip_member = f"Report_Data/{run_name}.zip"  # Optional report-data zip path.

if include_report_data_zip and report_zip_member in outer_members:  # Adds Report_Data zip only if requested.
    nested_zip_members.append(report_zip_member)  # Adds the Report_Data run zip.

nested_zip_members = list(dict.fromkeys(nested_zip_members))  # Removes duplicate zip paths while preserving order.

source_summary = pd.DataFrame({  # Creates a simple source summary table.
    "source_type": ["Direct CSV files", "Nested ZIP files"],
    "count": [len(direct_csv_members), len(nested_zip_members)],
})

display(source_summary)  # Displays source counts.

display(pd.DataFrame({"direct_csv_member": direct_csv_members}).head(20))  # Shows first direct CSVs.

display(pd.DataFrame({"nested_zip_member": nested_zip_members}))  # Shows nested zip files that will be processed.

In [ ]:
# CELL 4 Consolidate CSV files in excel workbook

audit_rows = []  # Stores audit records for every CSV attempted.

used_sheet_names = set(["_audit"])  # Reserves the audit sheet name.

data_sheet_count = 0  # Counts CSV sheets successfully written.

with pd.ExcelWriter(output_excel, engine="openpyxl") as writer:  # Opens the final Excel workbook for writing.
    with zipfile.ZipFile(transfer_zip, "r") as outer_zip:  # Opens the main transfer zip.
        for csv_member in direct_csv_members:  # Processes direct CSV files in Run_G0001.
            source_location = f"{transfer_zip}!{csv_member}"  # Records exact source path.
            before_count = len(audit_rows)  # Tracks audit count before writing.
            write_csv_to_workbook(outer_zip, csv_member, source_location, writer, audit_rows, used_sheet_names)  # Writes direct CSV.
            if len(audit_rows) > before_count and audit_rows[-1]["status"] == "WRITTEN":  # Checks whether the CSV was written.
                data_sheet_count += 1  # Adds to successful sheet count.

        for nested_zip_member in nested_zip_members:  # Processes zip files inside Run_G0001.
            print("Copying nested zip:", nested_zip_member)  # Shows progress before copying.
            nested_label = f"{transfer_zip}!{nested_zip_member}"  # Records nested zip source label.
            nested_temp_path = copy_zip_member_to_temp(outer_zip, nested_zip_member, str(transfer_zip))  # Copies nested zip to temp file.

            try:
                before_count = len(audit_rows)  # Tracks audit count before processing nested zip.
                process_zip_file(nested_temp_path, nested_label, writer, audit_rows, used_sheet_names)  # Writes CSVs from nested zip.
                new_records = audit_rows[before_count:]  # Gets audit records created from this nested zip.
                data_sheet_count += sum(record["status"] == "WRITTEN" for record in new_records)  # Counts written sheets.
            finally:
                if not keep_temporary_archives and nested_temp_path.exists():  # Checks whether temporary zip should be removed.
                    nested_temp_path.unlink()  # Deletes the temporary nested zip.

    audit_table = pd.DataFrame(audit_rows)  # Converts audit list into a table.

    if audit_table.empty:  # Handles case where no CSV files were found.
        audit_table = pd.DataFrame([{"status": "NO_CSV_FOUND", "message": f"No CSV files found for {run_name}."}])  # Creates audit message.

    audit_table.to_excel(writer, sheet_name="_Audit", index=False)  # Saves audit table inside the same workbook.

print("Workbook saved:", output_excel)  # Prints final workbook path.

print("CSV sheets written:", data_sheet_count)  # Prints how many CSV sheets were saved.

display(audit_table.head(30))  # Displays the first audit rows.

In [ ]:
# CELL 5 Check results

audit_check = pd.read_excel(output_excel, sheet_name="_Audit")  # Reads the audit sheet from the saved workbook.

status_summary = audit_check["status"].value_counts().reset_index()  # Counts each processing status.

status_summary.columns = ["status", "count"]  # Gives readable column names.

display(status_summary)  # Shows how many files were written, skipped, or errored.

display(audit_check.head(50))  # Shows the first 50 audit records.